# Pixel → World, and measuring the real world in a photo

`frame.unproject(px, py)` lifts any pixel to a 3D world coordinate using LiDAR
depth. `cyvl.measure` chains two unprojects into a real-world measurement —
sign heights, lane widths, setback distances, straight from one photo.

In [ ]:
import cyvl

scene = cyvl.load_scene("somerville")
frame = scene.nearest_frame(-71.093162, 42.387389)
lidar = frame.lidar(radius_m=25)   # fetch once, reuse for many queries
frame.show(overlay=frame.project(lidar.xyz, colors=lidar.rgb))

In [ ]:
# Click coordinates from the image above (pixel x, y)
from cyvl.geometry import utm_to_lonlat

point_3d = frame.unproject(1920, 1400, lidar=lidar)
lon, lat, alt = utm_to_lonlat(point_3d[None, :], frame.utm_epsg)[0]
print(f"pixel (1920, 1400) -> UTM {point_3d.round(2)} -> {lat:.6f}, {lon:.6f}")


In [ ]:
# Measure between two pixels — e.g. across a crosswalk
m = cyvl.measure(frame, (1500, 1500), (2400, 1500), lidar=lidar)
print(m)            # Measurement(X.XX m / Y.YY ft)
m.point_a, m.point_b

Cross-check against the data: project a sign from the signs layer into the photo and measure to it.

In [ ]:
import numpy as np

# Cross-check against the data: pick any sign from the inventory, find the
# photo that sees it, and project all nearby signs into that photo. (Sign
# coverage is partial — south/southeast only — so go sign -> photo, not
# photo -> signs.)
signs = scene.signs
sign = signs.iloc[42]
sign_frame = scene.nearest_frame(sign.geometry.x, sign.geometry.y)

lonlat = np.column_stack([signs.geometry.x, signs.geometry.y])
proj = sign_frame.project(lonlat=lonlat, alt=sign_frame.position[2])
nearby = proj.in_view & (proj.depth < 30)   # in the frustum AND close enough to see
visible = signs[nearby].assign(
    depth_m=proj.depth[nearby].round(1),
    pixel_x=proj.pixels[nearby, 0].round(0),
    pixel_y=proj.pixels[nearby, 1].round(0),
)
visible[["mutcd", "category", "condition", "depth_m", "pixel_x", "pixel_y"]]
